In [2]:
from autogen_agentchat.teams import RoundRobinGroupChat 
from autogen_agentchat.agents import AssistantAgent 
from autogen_ext.models.openai import OpenAIChatCompletionClient 
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
# MaxMessageTermination : groub chat의 메시지가 50개가 되면 chat room을 종료함(예시)
# TextMentionTermination : 예를들어 critic agent가  특정한 단어를 말하면 터미널 종료됨
from autogen_agentchat.ui import Console # 실시간 메시지 확인 가능

In [3]:
model = OpenAIChatCompletionClient(model="gpt-4o-mini")

clarity_agent = AssistantAgent(
    "ClarityAgent",
    model_client=model,
    system_message="""당신은 명확성과 단순함에 집중하는 전문 편집자입니다.
            당신의 임무는 모호함과 중복을 제거하고, 모든 문장을 간결하고 분명하게 만드는 것입니다.
            설득력이나 톤은 신경 쓰지 말고, 메시지가 읽기 쉽고 이해하기 쉬운지에만 집중하세요.""",
)

tone_agent = AssistantAgent(
    "ToneAgent",
    model_client=model,
    system_message="""당신은 감정 톤과 전문성에 집중하는 커뮤니케이션 코치입니다.
            당신의 임무는 이메일이 따뜻하고 자신감 있으며 인간적으로 느껴지도록 하되,
            대상에 맞는 전문성과 적절함을 유지하는 것입니다.
            감정적 공감을 높이고 표현을 다듬으며,
            딱딱하거나 차갑게 느껴지거나 지나치게 캐주얼한 표현을 조정하세요.""",
)

persuasion_agent = AssistantAgent(
    "PersuasionAgent",
    model_client=model,
    system_message="""당신은 마케팅, 행동 심리학, 카피라이팅에 훈련된 설득 전문가입니다.
            당신의 임무는 이메일의 설득력을 강화하는 것입니다.
            행동 유도를 개선하고, 논리 구조를 강화하며, 핵심 이점을 강조하세요.
            약하거나 수동적인 표현은 제거하세요.""",
)

synthesizer_agent = AssistantAgent(
    "SynthesizerAgent",
    model_client=model,
    system_message="""당신은 고급 이메일 작성 전문가입니다.
            당신의 역할은 이전 에이전트들의 모든 응답과 수정안을 읽고,
            가장 좋은 아이디어를 **종합하여** 하나의 완성도 높은 이메일 초안을 만드는 것입니다.
            다음에 집중하세요:
            명확성, 톤, 설득 요소의 통합;
            일관성, 유창함, 자연스러운 문체;
            전문적이고 효과적이며 읽기 쉬운 최종 버전 작성.""",
)

critic_agent = AssistantAgent(
    "CriticAgent",
    model_client=model,
    system_message="""당신은 이메일 품질 평가자입니다.
            당신의 임무는 종합된 이메일을 최종 검토하여
            전문적인 기준을 충족하는지 판단하는 것입니다.
            다음 요소를 검토하세요:
            명확성과 흐름, 적절한 전문적 톤, 효과적인 행동 유도, 전반적인 일관성.
            건설적이되 단호하게 평가하세요.
            이메일에 중대한 문제가 있다면
            (메시지가 불분명하거나, 비전문적인 톤이거나, 핵심 요소가 누락된 경우)
            **하나의 구체적인 개선 제안만** 제공하세요.
            이메일이 전문 기준을 충족하고 효과적으로 전달된다면
            'The email meets professional standards.'라고 응답한 뒤
            다음 줄에 `TERMINATE`를 작성하세요.
            충분히 완벽한 이메일만 승인해야 하며, 타협하지 마세요.""",
)


In [4]:
# terminate 조건 
text_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=30)

termination_condition = text_termination | max_messages_termination

In [7]:
# team을 assemble 함 
team = RoundRobinGroupChat(
    participants=[
        clarity_agent,
        tone_agent,
        persuasion_agent,
        synthesizer_agent,
        critic_agent # 순서중요
    ],
    termination_condition=termination_condition,
)

await Console(
    team.run_stream(task="안녕, 나 배고프니깐 밥좀 사줘 그리고 내 사업에 투자해주면 좋겠어 고마워")
    # run: 모든 메시지가 발생할때까지 기다린다. / run_stream : 메시지 발생 즉시 확인 가능
)

---------- TextMessage (user) ----------
안녕, 나 배고프니깐 밥좀 사줘 그리고 내 사업에 투자해주면 좋겠어 고마워
---------- TextMessage (ClarityAgent) ----------
안녕하세요. 식사를 제공할 수는 없지만, 사업에 투자하는 것에 대해 이야기할 수 있습니다. 어떤 사업인지 설명해 주시겠어요?
---------- TextMessage (ToneAgent) ----------
안녕하세요! 배고프시다니, 기분이 어떨지 이해가 갑니다. 따뜻한 식사를 하실 수 있기를 바랍니다. 

그리고 사업에 대한 투자 요청, 정말 감사합니다. 어떤 아이디어를 구상하고 계신지 듣고 싶습니다. 함께 이야기 나눠보는 기회를 가지면 좋을 것 같습니다. 고맙습니다!
---------- TextMessage (PersuasionAgent) ----------
안녕하세요! 배고프시다니 안타깝네요. 따뜻한 식사가 꼭 필요할 때죠.

이제 사업에 대한 당신의 투자 요청에 대해 이야기해 보겠습니다. 어떤 아이디어를 구상하고 계신가요? 시장에서 어떤 문제를 해결하고자 하는지, 그리고 그로 인해 어떤 가치를 창출할 수 있는지를 구체적으로 설명해 주시면 좋겠습니다.

당신의 비전을 듣고 함께 발전할 기회를 만들어보기를 기대합니다. 더 많은 이야기를 나눌 수 있기를 바랍니다. 고맙습니다!
---------- TextMessage (SynthesizerAgent) ----------
안녕하세요!

배고프시다니 안타깝네요. 따뜻한 식사가 필요한 순간인 것 같습니다.

이제 귀하의 사업에 대한 투자 요청에 대해 이야기해보면 좋겠습니다. 어떤 아이디어를 구상하고 계신지 궁금합니다. 귀하의 사업이 어떤 문제를 해결하고, 이를 통해 어떤 가치를 창출할 수 있는지 구체적으로 설명해 주시면 좋겠습니다.

귀하의 비전을 듣고 함께 발전할 기회를 만들어 나가기를 기대합니다. 더 많은 이야기를 나눌 수 있기를 바랍니다. 고맙습니다!
---------- Tex

TaskResult(messages=[TextMessage(id='e83a9b5c-d8e9-477e-92c9-5509e7e2af53', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 12, 20, 5, 31, 58, 450353, tzinfo=datetime.timezone.utc), content='안녕, 나 배고프니깐 밥좀 사줘 그리고 내 사업에 투자해주면 좋겠어 고마워', type='TextMessage'), TextMessage(id='967960c1-9bff-4582-b197-ee27128730c8', source='ClarityAgent', models_usage=RequestUsage(prompt_tokens=269, completion_tokens=34), metadata={}, created_at=datetime.datetime(2025, 12, 20, 5, 31, 59, 483583, tzinfo=datetime.timezone.utc), content='안녕하세요. 식사를 제공할 수는 없지만, 사업에 투자하는 것에 대해 이야기할 수 있습니다. 어떤 사업인지 설명해 주시겠어요?', type='TextMessage'), TextMessage(id='cf2b4371-b9a2-4156-be5a-c440119e6c08', source='ToneAgent', models_usage=RequestUsage(prompt_tokens=518, completion_tokens=80), metadata={}, created_at=datetime.datetime(2025, 12, 20, 5, 32, 2, 77698, tzinfo=datetime.timezone.utc), content='안녕하세요! 배고프시다니, 기분이 어떨지 이해가 갑니다. 따뜻한 식사를 하실 수 있기를 바랍니다. \n\n그리고 사업에 대한 투자 요청, 정말 감사합니다. 어떤 아이디어를 구상하고